In [18]:
import os
import json
import time
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

In [19]:
csv_df = pd.read_csv("cleaned_data.csv")

json_records = []

with open("cleaned_data.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        json_records.append(json.loads(line))

json_df = pd.DataFrame(json_records)

In [20]:
print("CSV Shape :", csv_df.shape)
print("JSONL Shape :", json_df.shape)

assert len(csv_df) == len(json_df), "Row count mismatch!"

print("\nLabel Distribution CSV:")
print(csv_df["target"].value_counts())

print("\nLabel Distribution JSONL:")
print(json_df["target"].value_counts())

CSV Shape : (11222, 3)
JSONL Shape : (11222, 3)

Label Distribution CSV:
target
0    9152
1    2070
Name: count, dtype: int64

Label Distribution JSONL:
target
0    9152
1    2070
Name: count, dtype: int64


HuggingFace Dataset

In [21]:
dataset = Dataset.from_pandas(csv_df)

train_test = dataset.train_test_split(test_size=0.2,seed=42)

# validation
val_test = train_test["test"].train_test_split(test_size=0.5,seed=42)

dataset_dict = DatasetDict({
    "train": train_test["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['cleaned_text', 'target', 'token_length'],
        num_rows: 8977
    })
    validation: Dataset({
        features: ['cleaned_text', 'target', 'token_length'],
        num_rows: 1122
    })
    test: Dataset({
        features: ['cleaned_text', 'target', 'token_length'],
        num_rows: 1123
    })
})


Tokenizers

In [22]:
tokenizers = {
    "bert": "bert-base-uncased",
    "distilbert": "distilbert-base-uncased",
    "roberta": "roberta-base"
}

results = []


In [23]:
def tokenize_function_dynamic(examples, tokenizer):
    return tokenizer(
        examples["cleaned_text"],
        truncation=True
    )

def tokenize_function_fixed(examples, tokenizer):
    return tokenizer(
        examples["cleaned_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


In [28]:
for name, model_name in tokenizers.items():

    print(f"Tokenizer : {name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Dynamic Padding

    start = time.time()

    tokenized_dynamic = dataset_dict.map(lambda x: tokenize_function_dynamic(x, tokenizer),batched=True)

    dynamic_time = time.time() - start

    # Fixed Padding

    start = time.time()

    tokenized_fixed = dataset_dict.map(lambda x: tokenize_function_fixed(x, tokenizer),batched=True)

    fixed_time = time.time() - start

    # Metrics


    token_lengths = []

    truncated = 0
    total = 0

    for item in tokenized_dynamic["train"]:

        length = len(item["input_ids"])
        token_lengths.append(length)

        if length >= 128:
            truncated += 1

        total += 1

    avg_length = np.mean(token_lengths)

    trunc_percent = (truncated / total) * 100

    # Sample Tokenized

    print("Sample Tokenized:\n")

    for i in range(5):
        print(tokenized_dynamic["train"][i])
        print()

    save_path = f"tokenized/{name}_dataset"

    tokenized_fixed.save_to_disk(save_path)

    tokenized_fixed.set_format( type="torch",columns=["input_ids", "attention_mask", "target"])

    torch.save( tokenized_fixed,os.path.join(save_path, f"{name}_dataset.pt"))


    results.append({
        "Tokenizer": name,
        "Dynamic Speed (s)": round(dynamic_time, 2),
        "Fixed Speed (s)": round(fixed_time, 2),
        "Avg Length": round(avg_length, 2),
        "% Truncated": round(trunc_percent, 2)
    })


    # Plot Histogram


    plt.figure(figsize=(8, 5))
    plt.hist(token_lengths, bins=30)

    plt.xlabel("Token Length")
    plt.ylabel("Frequency")
    plt.title(f"{name.upper()} Token Length Distribution")

    plt.savefig(f"tokenized/{name}_histogram.png")
    plt.close()

Tokenizer : bert


Map:   0%|          | 0/8977 [00:00<?, ? examples/s]

Map:   0%|          | 0/1122 [00:00<?, ? examples/s]

Map:   0%|          | 0/1123 [00:00<?, ? examples/s]

Map:   0%|          | 0/8977 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1123 [00:00<?, ? examples/s]

Sample Tokenized:

{'cleaned_text': 'iamreallysickandtiredofthemtryingtosidelinesocialistslikemethelpisabroadchurchthat', 'target': 0, 'token_length': 1, 'input_ids': [0, 6009, 20982, 29, 1758, 463, 90, 7651, 20168, 9506, 90, 15975, 90, 366, 43859, 35646, 1952, 3341, 5646, 19178, 354, 873, 14288, 23420, 6025, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

{'cleaned_text': 'indiademolisheskeralaskyscrapersoverenvironmentalviolationshttpstcobvqhaxaccpgoodstepbysupremecourhttpstcoqqzglcrih', 'target': 0, 'token_length': 1, 'input_ids': [0, 2028, 118, 33059, 1168, 10776, 330, 6653, 4970, 43104, 8645, 268, 7067, 2558, 49225, 337, 37834, 1635, 8166, 620, 438, 2413, 705, 1343, 298, 3631, 7904, 18188, 5715, 13975, 25372, 658, 5593, 3204, 2126, 8166, 620, 876, 31341, 329, 7210, 438, 1069, 298, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

Saving the dataset (0/1 shards):   0%|          | 0/8977 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1122 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1123 [00:00<?, ? examples/s]

comparison

In [31]:
results_df = pd.DataFrame(results)
results_df

,Tokenizer,Dynamic Speed (s),Fixed Speed (s),Avg Length,% Truncated
0,bert,6.98,8.60,23.67,0.0
1,distilbert,7.19,8.93,23.67,0.0
2,roberta,8.93,4.93,29.70,0.0
3,bert,3.36,5.24,23.67,0.0
4,distilbert,3.36,4.25,23.67,0.0
5,roberta,3.28,3.39,29.70,0.0
